# Module 3 - Class 6 Lab: Airbnb NYC End-to-End Data Prep

**Scenario:** Airbnb NYC - predict fair nightly price for new hosts  
**Lab guide:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/6](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/6)

---

## Objective
Turn the raw Inside Airbnb files into **`airbnb_clean.parquet`** with exactly **21 columns** and ~38,000 rows.  
This file is the input for every Airbnb module from M4 to M7.

## Time budget: 90 minutes

| Phase | Time | Task |
|-------|------|------|
| A | 10 min | Explore (3 charts) |
| 2 | 15 min | Clean price + bathrooms |
| 3 | 10 min | Impute missing |
| 4 | 5 min | Group neighbourhoods |
| 5 | 15 min | Haversine distance |
| 6 | 10 min | Date features |
| 7 | 10 min | Review features |
| 8 | 15 min | Assemble, validate, save |

---

## Required Output Schema (21 columns)

| # | Column | Type | Source |
|---|--------|------|--------|
| 1 | `listing_id` | int | listings (`id` renamed) |
| 2 | `host_id` | int | listings |
| 3 | `neighbourhood` | string | top-30 + Other |
| 4 | `borough` | string | `neighbourhood_group_cleansed` |
| 5 | `room_type` | string | listings |
| 6 | `latitude` | float | listings |
| 7 | `longitude` | float | listings |
| 8 | `accommodates` | int | listings |
| 9 | `bedrooms` | float | imputed by room_type median |
| 10 | `bathrooms` | float | parsed from `bathrooms_text` |
| 11 | `minimum_nights` | int | listings |
| 12 | `availability_365` | int | listings |
| 13 | `number_of_reviews` | int | listings |
| 14 | `reviews_per_year` | float | engineered |
| 15 | `review_scores_rating` | float | imputed with median |
| 16 | `host_age_days` | int | engineered from `host_since` |
| 17 | `distance_to_times_square_km` | float | Haversine |
| 18 | `mean_review_length` | float | engineered from reviews file |
| 19 | `description` | string | kept for M7 NLP |
| 20 | `price` | float | cleaned (no `$`, no `,`) |
| 21 | `log_price` | float | **TARGET** - `np.log1p(price)` |

## Setup - Google Colab + Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
LAB_DIR = '/content/drive/MyDrive/airbnb_lab'
os.makedirs(LAB_DIR + '/data', exist_ok=True)
os.chdir(LAB_DIR)
print('Working directory:', os.getcwd())

## Load Data

1. Go to https://insideairbnb.com/get-the-data/
2. Find **New York City, New York, United States** - latest scrape date
3. Copy the **detailed** `listings.csv.gz` and `reviews.csv.gz` URLs and paste below.

In [ ]:
import pandas as pd
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns

LISTINGS_URL = 'PASTE_LISTINGS_CSV_GZ_URL_HERE'
REVIEWS_URL  = 'PASTE_REVIEWS_CSV_GZ_URL_HERE'

for url, path in [(LISTINGS_URL, 'data/listings.csv.gz'),
                  (REVIEWS_URL,  'data/reviews.csv.gz')]:
    if not os.path.exists(path):
        print('Downloading', path, '...')
        urllib.request.urlretrieve(url, path)
    else:
        print('Already cached:', path)

listings = pd.read_csv('data/listings.csv.gz', low_memory=False)
reviews  = pd.read_csv('data/reviews.csv.gz')
print('listings :', listings.shape)
print('reviews  :', reviews.shape)

---

## Phase A - Exploratory Charts (10 min)

In [ ]:
# Chart 1 - Raw price format
print('First 10 price values:')
print(listings['price'].head(10).to_string())
print('dtype:', listings['price'].dtype)

In [ ]:
# Chart 2 - Room type + borough counts
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
listings['room_type'].value_counts().plot.bar(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Listings by room type')
axes[0].set_xlabel('Room type'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
listings['neighbourhood_group_cleansed'].value_counts().plot.bar(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Listings by borough')
axes[1].set_xlabel('Borough'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

In [ ]:
# Chart 3 - Missing values per column
missing = listings.isna().sum().sort_values(ascending=False).head(15)
plt.figure(figsize=(11, 4))
missing.plot.bar(color='salmon', edgecolor='white')
plt.title('Missing values per column (top 15)')
plt.ylabel('Missing count'); plt.xticks(rotation=35, ha='right')
plt.tight_layout(); plt.show()

---
## Stage 2 - Clean Price, Bathrooms & Dates (15 min)

**Decision log:**
- Price is stored as string `$150.00` -> strip `$` and `,`, convert to float
- Drop rows with missing/zero price (unusable for M4 model)
- Cap at 99th percentile (listings above ~$1k are outliers)
- Parse `bathrooms_text` -> float; map half-bath to 0.5
- Convert 4 date columns to datetime64

In [ ]:
listings['price'] = (
    listings['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
print('price dtype:', listings['price'].dtype, '  missing:', listings['price'].isna().sum())

In [ ]:
if 'bathrooms_text' in listings.columns:
    txt = listings['bathrooms_text'].fillna('').str.lower()
    is_half = txt.str.contains('half', regex=False)
    nums = txt.str.extract(r'(\d+\.?\d*)')[0].astype(float)
    listings['bathrooms'] = nums
    listings.loc[is_half & listings['bathrooms'].isna(), 'bathrooms'] = 0.5
elif 'bathrooms' not in listings.columns:
    listings['bathrooms'] = np.nan
print('bathrooms - min:', listings['bathrooms'].min(), ' max:', listings['bathrooms'].max(), ' missing:', listings['bathrooms'].isna().sum())

In [ ]:
date_cols = ['host_since', 'first_review', 'last_review', 'last_scraped']
for col in date_cols:
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        print(col, '->', listings[col].dtype)

In [ ]:
n0 = len(listings)
listings = listings.dropna(subset=['price'])
n1 = len(listings)
listings = listings[listings['price'] > 0]
n2 = len(listings)
price_cap = listings['price'].quantile(0.99)
listings = listings[listings['price'] <= price_cap]
n3 = len(listings)
print(f'raw:{n0:,}  no_nan:{n1:,}  no_zero:{n2:,}  no_outliers:{n3:,}')
print(f'Price cap (99th pct): ${price_cap:.2f}')

---
## Stage 3 - Impute Missing Values (10 min)

**Decision log:** `bedrooms` imputed by room_type median (Private room != Entire home). `bathrooms` and `review_scores_rating` imputed with overall median.

In [ ]:
listings['bedrooms'] = listings.groupby('room_type')['bedrooms'].transform(
    lambda x: x.fillna(x.median())
)
listings['bedrooms'] = listings['bedrooms'].fillna(listings['bedrooms'].median())
print('bedrooms missing after impute:', listings['bedrooms'].isna().sum())
print(listings.groupby('room_type')['bedrooms'].median())

In [ ]:
listings['bathrooms'] = listings['bathrooms'].fillna(listings['bathrooms'].median())
listings['review_scores_rating'] = listings['review_scores_rating'].fillna(
    listings['review_scores_rating'].median()
)
print('bathrooms missing:', listings['bathrooms'].isna().sum())
print('review_scores_rating missing:', listings['review_scores_rating'].isna().sum())

---
## Stage 4 - Group Rare Neighbourhoods (5 min)

**Decision log:** Keep top-30 by listing count, group rest as `Other`. One-hot encoding all 220+ would create sparse columns that overfit in M4.

In [ ]:
top30 = listings['neighbourhood_cleansed'].value_counts().head(30).index
listings['neighbourhood'] = listings['neighbourhood_cleansed'].where(
    listings['neighbourhood_cleansed'].isin(top30), 'Other'
)
print('Unique neighbourhoods:', listings['neighbourhood'].nunique())
print('Other count:', (listings['neighbourhood'] == 'Other').sum())

---
## Stage 5 - Haversine Distance to Times Square (15 min)

**Decision log:** Distance to Times Square is the strongest location signal. Haversine accounts for Earth's curvature - Euclidean on lat/lon degrees is wrong at NYC's latitude.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

TIMES_SQ_LAT, TIMES_SQ_LON = 40.7580, -73.9855
listings['distance_to_times_square_km'] = haversine(
    listings['latitude'].values, listings['longitude'].values,
    TIMES_SQ_LAT, TIMES_SQ_LON
)
print('distance_to_times_square_km  min:', round(listings['distance_to_times_square_km'].min(), 3),
      ' max:', round(listings['distance_to_times_square_km'].max(), 1), 'km')

In [ ]:
# Map scatter: NYC listings coloured by distance to Times Square
plt.figure(figsize=(8, 6))
sc = plt.scatter(listings['longitude'], listings['latitude'],
                 c=listings['distance_to_times_square_km'], s=1, cmap='RdYlGn_r', alpha=0.5)
plt.colorbar(sc, label='Distance to Times Square (km)')
plt.scatter(TIMES_SQ_LON, TIMES_SQ_LAT, c='blue', s=80, zorder=5, label='Times Square')
plt.legend(); plt.title('NYC listings coloured by distance to Times Square')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.tight_layout(); plt.show()

---
## Stage 6 - Date & Activity Features (10 min)

**Decision log:** `host_age_days` = experience proxy. `reviews_per_year` = activity proxy normalised by listing age.

In [ ]:
scrape_date = listings['last_scraped'].max()
print('Reference date:', scrape_date)

listings['host_age_days'] = ((scrape_date - listings['host_since']).dt.days
                             .fillna(0).astype(int))

years_active = (scrape_date - listings['first_review']).dt.days / 365.25
years_active = years_active.replace(0, np.nan)
listings['reviews_per_year'] = (listings['number_of_reviews'] / years_active).fillna(0).round(2)

print('host_age_days    max:', listings['host_age_days'].max())
print('reviews_per_year mean:', round(listings['reviews_per_year'].mean(), 2))

---
## Stage 7 - Review Text Features (10 min)

**Decision log:** `mean_review_length` signals listing quality. Listings with no reviews get 0 so M4 needs no special handling.

In [ ]:
review_agg = (
    reviews.dropna(subset=['comments'])
    .groupby('listing_id')['comments']
    .apply(lambda x: x.str.len().mean())
    .reset_index()
    .rename(columns={'comments': 'mean_review_length'})
)
listings = listings.merge(review_agg, left_on='id', right_on='listing_id', how='left')
if 'listing_id' in listings.columns:
    listings = listings.drop(columns=['listing_id'])
listings['mean_review_length'] = listings['mean_review_length'].fillna(0).round(1)
print('mean_review_length mean:', round(listings['mean_review_length'].mean(), 1), 'chars')

---
## Stage 8 - Assemble Final Dataset (15 min)

In [ ]:
listings['log_price'] = np.log1p(listings['price'])
print('log_price mean:', round(listings['log_price'].mean(), 4), '  (expect ~4.8)')

In [ ]:
COL_MAP = {
    'id': 'listing_id', 'host_id': 'host_id',
    'neighbourhood': 'neighbourhood',
    'neighbourhood_group_cleansed': 'borough',
    'room_type': 'room_type', 'latitude': 'latitude', 'longitude': 'longitude',
    'accommodates': 'accommodates', 'bedrooms': 'bedrooms', 'bathrooms': 'bathrooms',
    'minimum_nights': 'minimum_nights', 'availability_365': 'availability_365',
    'number_of_reviews': 'number_of_reviews', 'reviews_per_year': 'reviews_per_year',
    'review_scores_rating': 'review_scores_rating', 'host_age_days': 'host_age_days',
    'distance_to_times_square_km': 'distance_to_times_square_km',
    'mean_review_length': 'mean_review_length',
    'description': 'description', 'price': 'price', 'log_price': 'log_price',
}
missing_src = [c for c in COL_MAP if c not in listings.columns]
if missing_src: print('WARNING missing source columns:', missing_src)
out = listings[list(COL_MAP.keys())].rename(columns=COL_MAP).copy()
print('Final shape:', out.shape, '  (expect ~38000 rows x 21 cols)')

In [ ]:
int_cols   = ['listing_id','host_id','accommodates','minimum_nights',
              'availability_365','number_of_reviews','host_age_days']
float_cols = ['latitude','longitude','bedrooms','bathrooms','reviews_per_year',
              'review_scores_rating','distance_to_times_square_km',
              'mean_review_length','price','log_price']
str_cols   = ['neighbourhood','borough','room_type','description']
for col in int_cols:   out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0).astype(int)
for col in float_cols: out[col] = pd.to_numeric(out[col], errors='coerce').astype(float)
for col in str_cols:   out[col] = out[col].fillna('').astype(str)
print(out.dtypes)

In [ ]:
EXPECTED = list(COL_MAP.values())
print('Missing cols:', [c for c in EXPECTED if c not in out.columns] or 'None')
print('Extra cols  :', [c for c in out.columns if c not in EXPECTED] or 'None')
print('Column count:', len(out.columns), ' (expect 21)')
print()
print('NaN check:')
print(out[float_cols + int_cols].isna().sum())

In [ ]:
out_path = LAB_DIR + '/airbnb_clean.parquet'
out.to_parquet(out_path, index=False)
print('Saved:', out_path)
print('Rows               :', len(out))
print('Median price       : $', round(out['price'].median(), 2))
print('log_price mean     :', round(out['log_price'].mean(), 4), ' (expect ~4.8)')
print('Median distance km :', round(out['distance_to_times_square_km'].median(), 2))
print('Reviews covered    :', round((out['mean_review_length'] > 0).mean() * 100, 1), '%')

---
## Explanatory Chart for Marcus — Distance vs Price

In [ ]:
final = pd.read_parquet(out_path)
final['dist_bin'] = pd.cut(final['distance_to_times_square_km'],
                           bins=[0,2,5,10,15,35], labels=['0-2 km','2-5 km','5-10 km','10-15 km','15+ km'])
median_price = final.groupby('dist_bin', observed=True)['price'].median()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(median_price.index.astype(str), median_price.values, color='#FF5A5F', edgecolor='white', width=0.6)
for bar, val in zip(bars, median_price.values):
    ax.text(bar.get_x() + bar.get_width()/2, val+2, f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')
multiplier = round(median_price.iloc[0] / median_price.iloc[-1], 1)
ax.set_title(f'Median nightly price by distance to Times Square\nListings within 2 km charge {multiplier}x more than 15+ km away', fontsize=12)
ax.set_xlabel('Distance to Times Square', fontsize=11)
ax.set_ylabel('Median price per night (USD)', fontsize=11)
ax.set_ylim(0, median_price.max() * 1.2)
plt.tight_layout()
plt.savefig(LAB_DIR + '/marcus_chart_distance_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('Takeaway: distance to Times Square is the single strongest location signal.')

---
## Findings Report

*(Fill in after running all cells)*

### Final numbers
- Final row count: ___
- Median price: $___
- Median distance to Times Square: ___ km
- `log_price` mean: ___ (expect ~4.8)

### Top 3 insights
1. Location dominates price - listings near Times Square charge much more
2. Room type creates two markets - Entire home ~2x Private room
3. bedrooms had ~30% missing - imputed by room_type median, not global median

### One question for the M4 mentor
Should `reviews_per_year` be log-transformed? Very right-skewed distribution.

### Submission
Push `airbnb_clean.parquet`, this notebook, and `findings.md` to:
`module-3/class_6/submissions/CodeX/`